# 07 — Consultas analíticas em SQL (PostgreSQL / Supabase)

**Projeto:** Plataforma Financeira Integrada — Tríade (Linha 4)

**Objetivo:** responder às perguntas de negócio da Linha 4 diretamente no banco de dados, com SQL analítico (agregações, CTE e funções de janela), sobre a tabela `fato_anual_integrado`.

**Unidades:** valores monetários em R$ bilhões; indicadores em %.

In [1]:
from pathlib import Path
import tomllib
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
with open(ROOT / ".streamlit" / "secrets.toml", "rb") as arquivo:
    cfg = tomllib.load(arquivo)["connections"]["postgresql"]

engine = create_engine(URL.create(
    drivername="postgresql+psycopg2", username=cfg["username"], password=cfg["password"],
    host=cfg["host"], port=int(cfg["port"]), database=cfg["database"], query=cfg.get("query", {}),
), pool_pre_ping=True)


def consultar(sql):
    """Executa uma consulta SQL no Supabase e devolve um DataFrame."""
    return pd.read_sql(text(sql), engine)

In [2]:
consultar("""
    SELECT ano,
           ROUND(MAX(selic_media_pct)::numeric, 2)                  AS selic_media_pct,
           ROUND((-SUM(despesas_financeiras) / 1e9)::numeric, 1)    AS despesas_financeiras_bi,
           ROUND(AVG(divida_liquida_pl)::numeric, 2)                AS divida_liquida_pl_media,
           ROUND((SUM(lucro_liquido) / 1e9)::numeric, 1)            AS lucro_liquido_bi
    FROM fato_anual_integrado
    WHERE tipo = 'Não financeira'
    GROUP BY ano
    ORDER BY ano
""")

,ano,selic_media_pct,despesas_financeiras_bi,divida_liquida_pl_media,lucro_liquido_bi
0,2019,6.04,69.8,0.66,45.4
1,2020,2.89,115.1,1.46,34.9
2,2021,4.54,68.8,0.81,256.0
3,2022,12.55,38.7,0.54,330.8
4,2023,13.30,51.7,0.42,205.8


In [3]:
consultar("""
    SELECT setor, ano,
           ROUND((AVG(retorno_anual) * 100)::numeric, 1) AS retorno_medio_pct,
           ROUND(MAX(selic_media_pct)::numeric, 1)       AS selic_media_pct
    FROM fato_anual_integrado
    GROUP BY setor, ano
    ORDER BY setor, ano
""")

,setor,ano,retorno_medio_pct,selic_media_pct
0,Bancos,2019,8.3,6.0
1,Bancos,2020,-11.2,2.9
2,Bancos,2021,-31.4,4.5
3,Bancos,2022,24.2,12.5
4,Bancos,2023,42.3,13.3
5,Commodities Exportadoras,2019,5.5,6.0
6,Commodities Exportadoras,2020,59.2,2.9
7,Commodities Exportadoras,2021,3.8,4.5
8,Commodities Exportadoras,2022,5.0,12.5
9,Commodities Exportadoras,2023,6.1,13.3


In [4]:
consultar("""
    SELECT ano, ticker, setor,
           ROUND(MAX(ptax_variacao_ano_pct)::numeric, 1)  AS dolar_variacao_pct,
           ROUND((margem_ebit * 100)::numeric, 1)         AS margem_ebit_pct,
           ROUND((retorno_anual * 100)::numeric, 1)       AS retorno_pct
    FROM fato_anual_integrado
    WHERE setor IN ('Commodities Exportadoras', 'Exportadoras Industriais')
    GROUP BY ano, ticker, setor, margem_ebit, retorno_anual
    ORDER BY ano, ticker
""")

,ano,ticker,setor,dolar_variacao_pct,margem_ebit_pct,retorno_pct
0,2019,EMBR3,Exportadoras Industriais,4.4,-6.1,-8.4
1,2019,SUZB3,Commodities Exportadoras,4.4,10.1,3.9
2,2019,VALE3,Commodities Exportadoras,4.4,1.5,7.1
3,2019,WEGE3,Exportadoras Industriais,4.4,13.8,95.8
4,2020,EMBR3,Exportadoras Industriais,29.2,-8.6,-55.1
5,2020,SUZB3,Commodities Exportadoras,29.2,27.7,47.5
6,2020,VALE3,Commodities Exportadoras,29.2,25.0,70.9
7,2020,WEGE3,Exportadoras Industriais,29.2,16.1,120.3
8,2021,EMBR3,Exportadoras Industriais,8.1,4.7,180.5
9,2021,SUZB3,Commodities Exportadoras,8.1,44.4,2.7


In [5]:
consultar("""
    WITH por_empresa AS (
        SELECT ticker, setor,
               EXP(SUM(LN(1 + retorno_anual))) - 1 AS retorno_acumulado,
               AVG(roe)                            AS roe_medio,
               AVG(margem_ebit)                    AS margem_ebit_media,
               AVG(volatilidade_anual)             AS volatilidade_media
        FROM fato_anual_integrado
        GROUP BY ticker, setor
    )
    SELECT setor,
           ROUND((AVG(retorno_acumulado) * 100)::numeric, 1)   AS retorno_acumulado_pct,
           ROUND((AVG(roe_medio) * 100)::numeric, 1)           AS roe_medio_pct,
           ROUND((AVG(margem_ebit_media) * 100)::numeric, 1)   AS margem_ebit_media_pct,
           ROUND((AVG(volatilidade_media) * 100)::numeric, 1)  AS volatilidade_media_pct
    FROM por_empresa
    GROUP BY setor
    ORDER BY retorno_acumulado_pct DESC
""")

,setor,retorno_acumulado_pct,roe_medio_pct,margem_ebit_media_pct,volatilidade_media_pct
0,Energia,190.1,28.3,41.1,33.1
1,Exportadoras Industriais,171.9,8.9,7.8,41.8
2,Commodities Exportadoras,92.1,13.7,30.3,38.1
3,Bancos,16.6,16.0,NaN,31.6
4,Consumo/Varejo,-21.1,16.0,18.0,37.1


In [6]:
consultar("""
    SELECT ano, ticker, setor,
           ROUND((roe * 100)::numeric, 1)                          AS roe_pct,
           RANK() OVER (PARTITION BY ano ORDER BY roe DESC)        AS posicao
    FROM fato_anual_integrado
    WHERE roe IS NOT NULL
    ORDER BY ano, posicao
""")

,ano,ticker,setor,roe_pct,posicao
0,2019,EGIE3,Energia,33.0,1
1,2019,LREN3,Consumo/Varejo,23.4,2
2,2019,ABEV3,Consumo/Varejo,19.5,3
3,2019,ITUB4,Bancos,18.6,4
4,2019,WEGE3,Exportadoras Industriais,18.3,5
5,2019,PETR4,Energia,13.7,6
6,2019,VALE3,Commodities Exportadoras,-5.5,7
7,2019,EMBR3,Exportadoras Industriais,-8.9,8
8,2019,SUZB3,Commodities Exportadoras,-15.6,9
9,2020,EGIE3,Energia,36.1,1


## Interpretação — Pirâmide Invertida (Contexto → Insight → Ação)

**P1 — Selic × balanços**
- **Contexto:** a Selic média subiu de 2,9% (2020) para 13,3% (2023).
- **Insight:** as despesas financeiras das 8 empresas não financeiras **não acompanharam a Selic**: foram máximas em 2020 (R$ 115 bi), com o juro no piso, e menores em 2022–2023. A alavancagem média (Dívida Líquida/PL) caiu de 1,46 para 0,42. Hipótese explicativa: parte relevante da dívida da carteira é em dólar, e a variação cambial de 2020 (+29%) pesou mais que o juro doméstico; as empresas também desalavancaram antes do aperto.
- **Ação:** ao avaliar o risco de juros, separar a exposição à Selic da exposição cambial da dívida.

**P2 — Retorno setorial × Selic**
- **Insight:** Consumo/Varejo teve retorno médio negativo de 2020 a 2023; Energia (+22,0% e +62,5%) e Bancos (+24,2% e +42,3%) tiveram os melhores retornos em 2022–2023, com a Selic acima de 12%.
- **Ação:** em ciclos de aperto, setores sensíveis a crédito e consumo merecem cautela.

**P3 — Exportadoras e estresse cambial (2020: dólar +29,2%)**
- **Insight:** SUZB3 (+47,5%), VALE3 (+70,9%) e WEGE3 (+120,3%) subiram; a Suzano manteve margem EBIT de 27,7%, mas registrou prejuízo contábil (ROE −146%) pela reavaliação da dívida em dólar — efeito contábil, sem impacto direto no caixa operacional. A EMBR3 (−55,1%) foi exceção: a pandemia paralisou a aviação.
- **Ação:** distinguir proteção operacional (receita em dólar) de impacto contábil (dívida em dólar).

**P4 — Setor mais resiliente**
- **Insight:** Energia combinou o maior retorno acumulado (+190,1%), o maior ROE médio (28,3%), a maior margem EBIT (41,1%) e a menor volatilidade (33,1%).
- **Ação:** Energia foi o setor de melhor relação risco-retorno no período analisado.

**Limitações:** associação estatística, não causalidade; 1 a 2 empresas por setor; médias simples entre empresas; o ROE negativo da VALE3 em 2019 reflete o rompimento da barragem de Brumadinho, evento não macroeconômico.